# Phase 6 DAEAC FCBA RR NSV Hybrid Balanced Pseudo SVDB Smoke

Runs the balanced pseudo-label + collapse-guard Hybrid MK-MMD + MCC configs on the two SVDB adaptation scenarios. Defaults are smoke-sized; raise `EPOCHS` and remove sample caps for a real run.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess, sys
import numpy as np
import pandas as pd

os.environ['PYTHONUNBUFFERED'] = '1'
os.environ.setdefault('WANDB_MODE', 'disabled')

REPO_URL = 'https://github.com/YOUR_ORG/Best-thesis-in-council.git'
BRANCH = 'main'
EPOCHS = 2
MAX_SOURCE_SAMPLES = 1024
MAX_TARGET_SAMPLES = 1024
MAX_VAL_SAMPLES = 512

SCENARIOS = [
    ('ds1_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_mkmmd_mcc_balanced_ds1_svdb.yaml', 'daeac_fcba_rr_nsv_hybrid_mkmmd_mcc_balanced_ds1_svdb'),
    ('mitbih_svdb', 'configs/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_mkmmd_mcc_balanced_mitbih_svdb.yaml', 'daeac_fcba_rr_nsv_hybrid_mkmmd_mcc_balanced_mitbih_svdb'),
]

WORK = Path('/kaggle/working')
REPO = WORK / 'Best-thesis-in-council'
ECG = REPO / 'ecg_thesis'
print('Repo path:', ECG)

## 1. Clone Repo and Install

In [ ]:
if not ECG.exists():
    assert 'YOUR_ORG' not in REPO_URL, 'Edit REPO_URL before cloning.'
    subprocess.run(['git', 'clone', '--branch', BRANCH, REPO_URL, str(REPO)], check=True)
else:
    print('Repo already exists:', ECG)
assert ECG.exists(), ECG
os.chdir(ECG)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
subprocess.run([sys.executable, 'scripts/check_repo.py'], check=True)

## 2. Copy NSV RR Data Bundle

In [ ]:
candidate_dirs = [p.parent for p in Path('/kaggle/input').glob('**/record_split_manifest.json')]
print('Candidate record-split bundle dirs:')
for p in candidate_dirs:
    print(' -', p)
assert len(candidate_dirs) == 1, f'Expected one NSV bundle, found {candidate_dirs}'

dest = Path('data/processed/phase6_daeac_record_splits_nsv')
dest.mkdir(parents=True, exist_ok=True)
for src in sorted(candidate_dirs[0].glob('*.npz')):
    shutil.copy2(src, dest / src.name)
manifest = candidate_dirs[0] / 'record_split_manifest.json'
if manifest.exists():
    shutil.copy2(manifest, dest / manifest.name)

required = ['ds1_train.npz', 'ds1_val.npz', 'mitbih_train.npz', 'mitbih_val.npz', 'svdb_train.npz', 'svdb_val.npz', 'svdb_test.npz']
for name in required:
    path = dest / name
    assert path.exists(), path
    with np.load(path, allow_pickle=True) as data:
        assert data['class_names'].tolist() == ['N', 'S', 'V'], (name, data['class_names'].tolist())
        assert 'rr_features' in data.files, f'{name} has no rr_features.'
print('Copied NSV bundle to', dest)

## 3. Copy Required Source Pretrain Checkpoints

In [ ]:
checkpoints = {
    'daeac_fcba_rr_nsv_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_sqrtw/checkpoints/daeac_fcba_rr_nsv_sqrtw_base_best.pt'),
    'daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt': Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_mitbih_sqrtw/checkpoints/daeac_fcba_rr_nsv_mitbih_sqrtw_base_best.pt'),
}
for filename, target in checkpoints.items():
    candidates = sorted(Path('/kaggle/input').glob(f'**/{filename}'))
    print(filename, candidates)
    assert len(candidates) == 1, f'Expected one {filename}, found {candidates}'
    target.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(candidates[0], target)
    assert target.exists(), target

## 4. Validate Configs

In [ ]:
for scenario, config, _ in SCENARIOS:
    print('\nVALIDATE', scenario)
    subprocess.run([sys.executable, 'scripts/phase6_daeac_paper/00_validate_data.py', '--config', config], check=True)

## 5. Run Smoke Adaptation

In [ ]:
rows = []
for scenario, config, prefix in SCENARIOS:
    print('\n' + '=' * 88)
    print('BALANCED HYBRID SMOKE', scenario)
    print('=' * 88)
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_mcc/02_train_hybrid_mkmmd_mcc.py',
        '--config', config,
        '--epochs', str(EPOCHS),
        '--max-source-samples', str(MAX_SOURCE_SAMPLES),
        '--max-target-samples', str(MAX_TARGET_SAMPLES),
        '--max-val-samples', str(MAX_VAL_SAMPLES),
    ], check=True)

    summary_path = next(Path('outputs').glob(f'**/metrics/{prefix}_train_summary.json'))
    summary = json.loads(summary_path.read_text())
    hist = pd.DataFrame(summary['history'])
    hist_path = summary_path.parents[1] / f'{prefix}_history_smoke.csv'
    hist.to_csv(hist_path, index=False)
    display_cols = [c for c in ['epoch', 'loss', 'target_entropy', 'pseudo_selected', 'pseudo_guard_triggered', 'pseudo_guard_max_ratio', 'val_macro_f1', 'pseudo_counts', 'target_pred_counts'] if c in hist.columns]
    display(hist[display_cols])
    rows.append({
        'scenario': scenario,
        'best_epoch': summary.get('best_epoch'),
        'best_val_macro_f1': summary.get('best_val_macro_f1'),
        'latest_checkpoint': summary.get('latest_checkpoint'),
        'best_checkpoint': summary.get('best_checkpoint'),
        'history_path': str(hist_path),
    })

df = pd.DataFrame(rows)
display(df)
out = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_mkmmd_mcc_balanced_svdb_smoke_summary.csv')
df.to_csv(out, index=False)
print('Saved', out)

## 6. Optional Target Evaluation

In [ ]:
eval_rows = []
for scenario, config, prefix in SCENARIOS:
    summary_path = next(Path('outputs').glob(f'**/metrics/{prefix}_train_summary.json'))
    summary = json.loads(summary_path.read_text())
    best_checkpoint = Path(summary['best_checkpoint'])
    method = f'{prefix}_best_smoke'
    subprocess.run([
        sys.executable, '-u', 'scripts/phase6_daeac_paper/03_eval.py',
        '--config', config,
        '--checkpoint', str(best_checkpoint),
        '--method-name', method,
        '--dataset', 'target',
    ], check=True)
    metrics_path = summary_path.parents[1] / 'metrics' / f'{method}_target_test_metrics.json'
    metrics = json.loads(metrics_path.read_text())
    eval_rows.append({
        'scenario': scenario,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'N_f1': metrics['per_class']['N']['f1'],
        'S_f1': metrics['per_class']['S']['f1'],
        'V_f1': metrics['per_class']['V']['f1'],
        'metrics_path': str(metrics_path),
    })
eval_df = pd.DataFrame(eval_rows)
display(eval_df)
eval_out = Path('outputs/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_mkmmd_mcc_balanced_svdb_smoke_eval_summary.csv')
eval_df.to_csv(eval_out, index=False)
print('Saved', eval_out)

## 7. Package Outputs

In [ ]:
shutil.make_archive('/kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_balanced_svdb_smoke_outputs', 'zip', 'outputs')
print('Wrote /kaggle/working/phase6_daeac_fcba_latefusion_rr_nsv_hybrid_balanced_svdb_smoke_outputs.zip')